In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, concat, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType 

In [0]:
person_df = (
    spark.read
        .format("json")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/person.json")
)
display(person_df)

In [0]:
(
    person_df.select(
        col("personId").alias("person_id"),
        col("personName.forename").alias("forename"),
        col("personName.surname").alias("surname")
    )
    .withColumn("fullname",
        concat(col("forename"), lit(" "), col("surname"))
    )
    .show()
)

In [0]:
personName_shema = StructType([
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])
person_schema = StructType([
    StructField("personId", IntegerType(), True),
    StructField("personName", personName_shema, True)
])
person_df = (
    spark.read
        .schema(person_schema)
        .format("json")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/person.json")
)
person_df.printSchema()
display(person_df)

In [0]:
person_df = (
    add_ingestion_date(person_df)
        .select(
            col("personId").alias("person_id"),
            concat(col("personName.forename"), lit(" "), col("personName.surname")).alias("fullname"),
            lit(v_env).alias("env"),
            lit(v_file_date).alias("file_date")
).drop("personName")
)

display(person_df)

In [0]:
overwrite_partitioned_table("movie_silver", "persons", "file_date", v_file_date)

In [0]:
(
    person_df.write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable(f"movie_silver.persons")
)


In [0]:
dbutils.notebook.exit("Success")